# Step 1: Project Overview & Objectives
---
The goal of this notebook is to build a Named Entity Recognition (NER) pipeline to detect disease entities in medical text using the **BC5CDR dataset**. We are using the BIO tagging scheme (`B-Disease`, `I-Disease`, `O`).

To find the best architecture, we are benchmarking **15 different combinations**:
* **5 Embeddings:** Word2Vec, GloVe, FastText, ELMo, and PubMedBERT.
* **3 Tokenizers:** Whitespace, NLTK, and Hugging Face BPE.

The core pipeline runs every embedding through a custom **CharCNN + BiLSTM + CRF** architecture to classify labels at the token level.

In [ ]:
import subprocess, sys

def pip_install(pkg):
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", pkg],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

pip_install("numpy==1.26.4")
pip_install("scipy==1.12.0")
pip_install("gdown")
pip_install("transformers")
pip_install("datasets")
pip_install("seqeval")
pip_install("gensim")
pip_install("torch")
pip_install("pytorch-crf")
pip_install("tensorflow")
pip_install("tensorflow-hub")
pip_install("nltk")

print("✅ All packages installed. Restarting kernel now...")

import os
os.kill(os.getpid(), 9)

In [1]:
import warnings
warnings.filterwarnings("ignore")

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
print("✅ NLTK data downloaded!")

In [4]:
import os
import gdown

file_id = '1T7fUoG5010_J46jnzBoTgpEvk5zQTXhD'
url = f'https://drive.google.com/uc?id={file_id}'
output = 'data.zip'
print("Downloading dataset from Google Drive...")
gdown.download(url, output, quiet=False, fuzzy=True)
print("Extracting files...")
os.system(f"unzip -oq {output} -d /kaggle/working/")
if os.path.exists(output):
    os.remove(output)
expected_path = "/kaggle/working/"
if os.path.exists(expected_path):
    print(f"\n[SUCCESS] Data ready at: {expected_path}")
else:
    print("\n[ERROR] Directory not found. Check zip structure.")

Downloading...
From: https://drive.google.com/uc?id=1T7fUoG5010_J46jnzBoTgpEvk5zQTXhD
To: /kaggle/working/data.zip
100%|██████████| 787k/787k [00:00<00:00, 137MB/s]

Extracting files...

[SUCCESS] Data ready at: /kaggle/working/


In [5]:
# ============================================================
# BLOCK 1: All Imports
# ============================================================
import os, re, time, random, warnings
import numpy as np
import pandas as pd
from collections import defaultdict
warnings.filterwarnings('ignore')

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.cuda.amp import GradScaler, autocast

# Transformers
from transformers import (
    BertTokenizerFast, BertModel,
    AutoTokenizer, AutoModel,
    get_cosine_schedule_with_warmup,
    get_linear_schedule_with_warmup
)

# Gensim
import gensim
import gensim.downloader as gensim_api
from gensim.models import Word2Vec, FastText, KeyedVectors

# NLTK
import nltk
from nltk.tokenize import word_tokenize

# CRF
from torchcrf import CRF

# Metrics
from seqeval.metrics import f1_score, classification_report

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = torch.cuda.is_available()
print(f"Using device: {DEVICE}  |  AMP: {USE_AMP}")

Using device: cuda  |  AMP: True


In [6]:
# ============================================================
# BLOCK 2: Load BC5CDR CoNLL-format Data
# ============================================================
DATA_DIR = "/kaggle/working/"

def load_conll(filepath):
    sentences, tokens, tags = [], [], []
    with open(filepath, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line == "" or line.startswith("-DOCSTART-"):
                if tokens:
                    sentences.append((tokens, tags))
                    tokens, tags = [], []
            else:
                parts = line.split()
                tokens.append(parts[0])
                tags.append(parts[-1])
    if tokens:
        sentences.append((tokens, tags))
    return sentences

train_data = load_conll(os.path.join(DATA_DIR, "train.txt"))
val_data   = load_conll(os.path.join(DATA_DIR, "val.txt"))
test_data  = load_conll(os.path.join(DATA_DIR, "test.txt"))

print(f"Train sentences : {len(train_data)}")
print(f"Val   sentences : {len(val_data)}")
print(f"Test  sentences : {len(test_data)}")
print("\nSample sentence:")
print("Tokens:", train_data[0][0][:10])
print("Tags  :", train_data[0][1][:10])

Train sentences : 4560
Val   sentences : 4581
Test  sentences : 4797

Sample sentence:
Tokens: ['Selegiline', '-', 'induced', 'postural', 'hypotension', 'in', 'Parkinson', "'", 's', 'disease']
Tags  : ['O', 'O', 'O', 'B-Disease', 'I-Disease', 'O', 'B-Disease', 'I-Disease', 'I-Disease', 'I-Disease']


In [7]:
# ============================================================
# BLOCK 3: Build Tag Vocabulary
# ============================================================
all_tags = set()
for _, tags in train_data + val_data + test_data:
    all_tags.update(tags)

tag_list = sorted(all_tags)
tag2idx  = {tag: idx for idx, tag in enumerate(tag_list)}
idx2tag  = {idx: tag for tag, idx in tag2idx.items()}
NUM_TAGS = len(tag2idx)

print("Tag vocabulary:", tag2idx)
print("Number of tags:", NUM_TAGS)

Tag vocabulary: {'B-Disease': 0, 'I-Disease': 1, 'O': 2}
Number of tags: 3


# Step 2: Tokenization & Tag Alignment Strategy
---
When you change tokenizers (e.g., splitting words into subwords using BPE), the number of tokens in a sentence changes. This breaks the 1:1 alignment with our original ground-truth disease tags.

To fix this, our `retokenize_sentence` function does the following:
1. **Tracks word boundaries** across different tokenization styles.
2. **Maps the original BIO tags** to the new token fragments.
3. **Uses a binary mask array (`mask`)** for subwords, ensuring we only evaluate the model on the first subword of a split phrase so metrics don't get artificially inflated.

In [8]:
# ============================================================
# BLOCK 4: Define Three Tokenizers
# ============================================================

# Global list of tokenizers used across all experiments
TOKENIZERS = ["whitespace", "bpe", "nltk"]

def whitespace_tokenize(text):
    return text.split()

def nltk_tokenize(text):
    return word_tokenize(text)

# Cased tokenizer
bert_tokenizer = BertTokenizerFast.from_pretrained("bert-base-cased")

def hf_tokenize(text):
    return bert_tokenizer.tokenize(text)

def retokenize_sentence(tokens, tags, tokenizer_name="whitespace"):
    new_tokens, new_tags, mask = [], [], []
    sentence = " ".join(tokens)

    if tokenizer_name == "whitespace":
        for tok, tag in zip(tokens, tags):
            parts = tok.split()
            for i, p in enumerate(parts):
                new_tokens.append(p)
                new_tags.append(tag)
                mask.append(1 if i == 0 else 0)

    elif tokenizer_name == "nltk":
        nltk_toks = word_tokenize(sentence)
        new_tokens = nltk_toks
        joined = " ".join(tokens)
        orig_char_pos = 0
        for ntok in nltk_toks:
            pos = joined.find(ntok, orig_char_pos)
            char_count = 0
            found_tag = "O"
            for ti, t in enumerate(tokens):
                if char_count <= pos < char_count + len(t):
                    found_tag = tags[ti]
                    break
                char_count += len(t) + 1
            new_tags.append(found_tag)
            mask.append(1)
            orig_char_pos = max(0, pos - 1)

    elif tokenizer_name == "bpe":
        for tok, tag in zip(tokens, tags):
            sub = bert_tokenizer.tokenize(tok)
            if not sub:
                sub = ["[UNK]"]
            for i, s in enumerate(sub):
                new_tokens.append(s)
                new_tags.append(tag)
                mask.append(1 if i == 0 else 0)

    return new_tokens, new_tags, mask

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

# Step 3: Morphological Features via CharCNN
---
Medical texts have tons of highly specific jargon, suffixes (like *-itis* or *-emia*), and Out-of-Vocabulary (OOV) words that static embeddings miss. 

We add a character-level CNN layer to solve this:
1. Words are split into character sequences.
2. A 1D Convolutional layer extracts character-level patterns (prefixes/suffixes).
3. Max-pooling generates a fixed-size representation for each word.

This character vector is concatenated with the main word embedding, allowing the model to understand unknown words based on how they are spelled.

In [9]:
# ============================================================
# BLOCK 4B: Character Vocabulary for CharCNN
# ============================================================
# Build char vocab from training tokens for the character-level CNN.
# CharCNN captures morphological patterns (prefixes/suffixes) and
# handles OOV words — crucial for biomedical terms.

all_chars = set()
for toks, _ in train_data:
    for tok in toks:
        all_chars.update(tok.lower())

CHAR_PAD  = "<PAD>"
CHAR_UNK  = "<UNK>"
char_list = [CHAR_PAD, CHAR_UNK] + sorted(all_chars)
char2idx  = {c: i for i, c in enumerate(char_list)}
CHAR_VOCAB_SIZE = len(char2idx)
CHAR_PAD_IDX    = char2idx[CHAR_PAD]
MAX_WORD_LEN    = 30     # truncate very long tokens

print(f"Character vocabulary size: {CHAR_VOCAB_SIZE}")
print(f"Sample chars: {char_list[:20]}")

Character vocabulary size: 56
Sample chars: ['<PAD>', '<UNK>', '"', '%', "'", '(', ')', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7']


In [10]:
# ============================================================
# BLOCK 5: Download / Load Pre-trained Embeddings
# ============================================================
EMBED_DIM = 100

print("Training Word2Vec...")
w2v_corpus = [toks for toks, _ in train_data]
w2v_model  = Word2Vec(w2v_corpus, vector_size=EMBED_DIM,
                      window=5, min_count=1, workers=4, epochs=20)
w2v_wv     = w2v_model.wv
print(f"Word2Vec vocab size: {len(w2v_wv)}")

print("\nLoading GloVe 100d...")
glove_wv = gensim_api.load("glove-wiki-gigaword-100")
print(f"GloVe vocab size: {len(glove_wv)}")

print("\nTraining FastText...")
ft_model = FastText(w2v_corpus, vector_size=EMBED_DIM,
                    window=5, min_count=1, workers=4, epochs=20)
ft_wv    = ft_model.wv
print(f"FastText vocab size: {len(ft_wv)}")

print("\nEmbedding models ready.")

Training Word2Vec...
Word2Vec vocab size: 9981

Loading GloVe 100d...
[==================================================] 100.0% 128.1/128.1MB downloaded
GloVe vocab size: 400000

Training FastText...
FastText vocab size: 9981

Embedding models ready.


In [11]:
# ============================================================
# BLOCK 6: Build Word Vocabulary + Static Embedding Matrices
# ============================================================
all_train_tokens = [t.lower() for toks, _ in train_data for t in toks]
vocab    = ["<PAD>", "<UNK>"] + sorted(set(all_train_tokens))
word2idx = {w: i for i, w in enumerate(vocab)}
VOCAB_SIZE = len(word2idx)
PAD_IDX    = word2idx["<PAD>"]
UNK_IDX    = word2idx["<UNK>"]
print(f"Vocabulary size: {VOCAB_SIZE}")

def build_embedding_matrix(wv, word2idx, embed_dim):
    matrix = np.zeros((len(word2idx), embed_dim))
    hits = 0
    for word, idx in word2idx.items():
        if word in wv:
            matrix[idx] = wv[word]
            hits += 1
        else:
            matrix[idx] = np.random.uniform(-0.25, 0.25, embed_dim)
    print(f"  Coverage: {hits}/{len(word2idx)} ({100*hits/len(word2idx):.1f}%)")
    return torch.tensor(matrix, dtype=torch.float32)

print("Building Word2Vec matrix...")
w2v_matrix   = build_embedding_matrix(w2v_wv, word2idx, EMBED_DIM)
print("Building GloVe matrix...")
glove_matrix = build_embedding_matrix(glove_wv, word2idx, EMBED_DIM)
print("Building FastText matrix...")
ft_matrix    = build_embedding_matrix(ft_wv, word2idx, EMBED_DIM)

Vocabulary size: 8741
Building Word2Vec matrix...
  Coverage: 7693/8741 (88.0%)
Building GloVe matrix...
  Coverage: 7482/8741 (85.6%)
Building FastText matrix...
  Coverage: 8741/8741 (100.0%)


In [13]:
# ============================================================
# BLOCK 7: PyTorch Dataset — Static Embeddings + Char Features
# ============================================================
MAX_LEN = 128

def tokens_to_char_ids(tokens, char2idx, max_word_len=MAX_WORD_LEN):
    """Convert list of tokens to (len, max_word_len) char-id tensor."""
    result = []
    for tok in tokens:
        ids = [char2idx.get(c.lower(), char2idx["<UNK>"]) for c in tok[:max_word_len]]
        ids += [char2idx["<PAD>"]] * (max_word_len - len(ids))
        result.append(ids)
    return result   # list of lists


class NERDataset(Dataset):
    def __init__(self, data, word2idx, tag2idx, char2idx,
                 tokenizer_name="whitespace", max_len=MAX_LEN):
        self.samples = []
        for tokens, tags in data:
            toks, tgs, mask = retokenize_sentence(tokens, tags, tokenizer_name)
            toks, tgs, mask = toks[:max_len], tgs[:max_len], mask[:max_len]
            ids      = [word2idx.get(t.lower(), UNK_IDX) for t in toks]
            char_ids = tokens_to_char_ids(toks, char2idx)
            tids     = [tag2idx[g] for g in tgs]
            self.samples.append((ids, char_ids, tids, mask))

    def __len__(self): return len(self.samples)
    def __getitem__(self, i): return self.samples[i]


def collate_fn(batch):
    ids_list, char_list, tags_list, mask_list = zip(*batch)
    lengths  = [len(x) for x in ids_list]
    max_l    = max(lengths)
    mwl      = MAX_WORD_LEN

    ids_pad  = torch.zeros(len(batch), max_l, dtype=torch.long)
    char_pad = torch.zeros(len(batch), max_l, mwl, dtype=torch.long)
    tags_pad = torch.full((len(batch), max_l), -1, dtype=torch.long)
    mask_pad = torch.zeros(len(batch), max_l, dtype=torch.bool)

    for i, (ids, chars, tags, msk) in enumerate(zip(ids_list, char_list, tags_list, mask_list)):
        l = len(ids)
        ids_pad[i, :l]    = torch.tensor(ids)
        char_pad[i, :l]   = torch.tensor(chars)
        tags_pad[i, :l]   = torch.tensor(tags)
        mask_pad[i, :l]   = torch.tensor(msk, dtype=torch.bool)

    att_mask = (ids_pad != 0)
    return ids_pad, char_pad, tags_pad, att_mask, mask_pad

In [14]:
# ============================================================
# BLOCK 7B: Dataset for PubMedBERT
# ============================================================

class BERTNERDataset(Dataset):
    def __init__(self, data, tag2idx, tokenizer,
                 tokenizer_name="bpe", max_len=MAX_LEN):
        self.samples   = []
        self.tokenizer = tokenizer
        for tokens, tags in data:
            toks, tgs, mask = retokenize_sentence(tokens, tags, tokenizer_name)
            encoding = self.tokenizer(
                toks,
                is_split_into_words=True,
                return_offsets_mapping=False,
                padding='max_length',
                truncation=True,
                max_length=max_len,
                return_tensors='pt'
            )
            input_ids      = encoding['input_ids'].squeeze(0)
            attention_mask = encoding['attention_mask'].squeeze(0)
            word_ids       = encoding.word_ids()

            label_ids = []
            prev_word_id = None
            for wid in word_ids:
                if wid is None:
                    label_ids.append(-1)
                elif wid != prev_word_id:
                    label_ids.append(tag2idx[tgs[wid]] if wid < len(tgs) else -1)
                else:
                    label_ids.append(-1)
                prev_word_id = wid

            self.samples.append((input_ids, attention_mask,
                                 torch.tensor(label_ids, dtype=torch.long)))

    def __len__(self): return len(self.samples)
    def __getitem__(self, i): return self.samples[i]

def bert_collate_fn(batch):
    ids, masks, labels = zip(*batch)
    return (torch.stack(ids), torch.stack(masks), torch.stack(labels))

In [15]:
# ============================================================
# BLOCK 8: ELMo Contextual Embeddings via TensorFlow Hub
# ============================================================
import tensorflow as tf
import tensorflow_hub as hub

print("Loading ELMo from TF Hub...")
elmo_tf = hub.load("https://tfhub.dev/google/elmo/3")
ELMO_DIM = 1024
print("ELMo loaded successfully.")

def get_elmo_embeddings(sentences_tokens, batch_size=32):
    all_embeddings = []
    for i in range(0, len(sentences_tokens), batch_size):
        batch   = sentences_tokens[i:i+batch_size]
        lengths = [len(toks) for toks in batch]
        texts   = tf.constant([" ".join(toks) for toks in batch])
        outputs = elmo_tf.signatures["default"](text=texts)
        emb_np  = outputs["elmo"].numpy()
        for j, length in enumerate(lengths):
            all_embeddings.append(
                torch.tensor(emb_np[j, :length, :], dtype=torch.float32))
    return all_embeddings

2026-06-02 05:47:53.385973: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780379273.866706     174 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780379273.982022     174 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780379275.027428     174 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780379275.027474     174 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780379275.027477     174 computation_placer.cc:177] computation placer alr

Loading ELMo from TF Hub...


I0000 00:00:1780379296.489281     174 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1780379296.495533     174 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


ELMo loaded successfully.


In [16]:
# ============================================================
# BLOCK 8B: ELMo Dataset — pre-compute embeddings per sentence
# ============================================================
def clean_tokens_for_elmo(tokens):
    """Strip WordPiece ## prefixes so ELMo receives natural word strings."""
    return [tok.lstrip("##") if tok.startswith("##") else tok for tok in tokens]

class ELMoNERDataset(Dataset):
    def __init__(self, data, tag2idx, tokenizer_name="whitespace",
                 max_len=MAX_LEN, batch_size=32):
        self.embeddings, self.tags_list, self.masks_list = [], [], []
        all_tokens, all_tags_r, all_masks = [], [], []
        for tokens, tags in data:
            toks, tgs, msk = retokenize_sentence(tokens, tags, tokenizer_name)
            toks, tgs, msk = toks[:max_len], tgs[:max_len], msk[:max_len]
            # ✨ Strip ## for ELMo — BPE split still drives tag alignment & mask
            elmo_toks = clean_tokens_for_elmo(toks)
            all_tokens.append(elmo_toks)      # <-- cleaned tokens go to ELMo
            all_tags_r.append([tag2idx[g] for g in tgs])
            all_masks.append(msk)

        print(f"  Running ELMo on {len(all_tokens)} sentences... (tokenizer={tokenizer_name})")
        all_embs = get_elmo_embeddings(all_tokens, batch_size=batch_size)
        for j, emb in enumerate(all_embs):
            self.embeddings.append(emb)
            self.tags_list.append(all_tags_r[j])
            self.masks_list.append(all_masks[j])
        print(f"  Done. {len(self.embeddings)} sentences embedded.")

    def __len__(self): return len(self.embeddings)
    def __getitem__(self, i):
        return self.embeddings[i], self.tags_list[i], self.masks_list[i]

def elmo_collate_fn(batch):
    embs, tags, masks = zip(*batch)
    lengths = [e.size(0) for e in embs]
    max_l   = max(lengths)
    B, D    = len(embs), embs[0].size(-1)
    emb_pad  = torch.zeros(B, max_l, D)
    tags_pad = torch.full((B, max_l), -1, dtype=torch.long)
    mask_pad = torch.zeros(B, max_l, dtype=torch.bool)
    att_mask = torch.zeros(B, max_l, dtype=torch.bool)
    for i, (e, t, m) in enumerate(zip(embs, tags, masks)):
        l = e.size(0)
        emb_pad[i, :l]  = e
        tags_pad[i, :l] = torch.tensor(t)
        mask_pad[i, :l] = torch.tensor(m, dtype=torch.bool)
        att_mask[i, :l] = True
    return emb_pad, tags_pad, att_mask, mask_pad

# Step 4: Sequence Modeling with BiLSTM-CRF
---
I used a combined **BiLSTM-CRF** setup instead of a standard Softmax classifier:

* **BiLSTM:** Takes our word + character embeddings and processes the sequence forward and backward to capture left-and-right context around every word.
* **CRF Layer:** Standard classifiers predict each tag independently, which allows illegal sequences (like an `I-Disease` tag right after an `O` tag). The CRF layer models sequence transitions globally by calculating path scores using a learnable transition matrix.

In [17]:
# ============================================================
# BLOCK 9: CharCNN + BiLSTM-CRF (Static Embeddings)
# ============================================================

class CharCNN(nn.Module):
    """Character-level CNN producing a fixed-size representation per token."""
    def __init__(self, char_vocab_size, char_embed_dim=30,
                 num_filters=50, kernel_size=3, char_pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(char_vocab_size, char_embed_dim,
                                      padding_idx=char_pad_idx)
        self.conv = nn.Conv1d(char_embed_dim, num_filters,
                              kernel_size=kernel_size, padding=kernel_size // 2)
        self.out_dim = num_filters

    def forward(self, char_ids):
        # char_ids: (B, seq_len, max_word_len)
        B, S, W = char_ids.shape
        x = self.embedding(char_ids.view(B * S, W))   # (B*S, W, dim)
        x = x.transpose(1, 2)                          # (B*S, dim, W)
        x = F.relu(self.conv(x))                       # (B*S, filters, W)
        x = x.max(dim=2).values                        # (B*S, filters) — max-pool
        return x.view(B, S, -1)                        # (B, S, filters)


class BiLSTMCRF(nn.Module):
    """BiLSTM-CRF with word + character embeddings."""
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_tags, pad_idx,
                 char_vocab_size, char_embed_dim=30, char_filters=50,
                 embedding_matrix=None, freeze_embed=False, dropout=0.5):
        super().__init__()
        # Word embedding
        self.word_embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        if embedding_matrix is not None:
            self.word_embedding.weight = nn.Parameter(embedding_matrix)
            self.word_embedding.weight.requires_grad = not freeze_embed

        # Character CNN
        self.char_cnn = CharCNN(char_vocab_size, char_embed_dim,
                                char_filters, kernel_size=3,
                                char_pad_idx=CHAR_PAD_IDX)

        lstm_input_dim = embed_dim + char_filters   # concatenate word + char

        self.dropout = nn.Dropout(dropout)
        self.bilstm  = nn.LSTM(lstm_input_dim, hidden_dim // 2,
                               num_layers=2, batch_first=True,
                               bidirectional=True, dropout=dropout)
        self.fc  = nn.Linear(hidden_dim, num_tags)
        self.crf = CRF(num_tags, batch_first=True)

    def forward(self, input_ids, char_ids, attention_mask, tags=None):
        word_emb = self.dropout(self.word_embedding(input_ids))  # (B, S, E)
        char_emb = self.dropout(self.char_cnn(char_ids))          # (B, S, C)
        x = torch.cat([word_emb, char_emb], dim=-1)               # (B, S, E+C)
        out, _ = self.bilstm(x)
        emissions = self.fc(self.dropout(out))

        if tags is not None:
            tags_crf = tags.clone()
            tags_crf[tags_crf == -1] = 0
            return -self.crf(emissions, tags_crf, mask=attention_mask, reduction='mean')
        return self.crf.decode(emissions, mask=attention_mask)

In [18]:
# ============================================================
# BLOCK 9B: BiLSTM-CRF for Pre-computed Embeddings (ELMo)
# ============================================================
class PrecomputedBiLSTMCRF(nn.Module):
    def __init__(self, embed_dim, hidden_dim, num_tags, dropout=0.5):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.bilstm  = nn.LSTM(embed_dim, hidden_dim // 2,
                               num_layers=2, batch_first=True,
                               bidirectional=True, dropout=dropout)
        self.fc  = nn.Linear(hidden_dim, num_tags)
        self.crf = CRF(num_tags, batch_first=True)

    def forward(self, embeddings, attention_mask, tags=None):
        out, _    = self.bilstm(self.dropout(embeddings))
        emissions = self.fc(self.dropout(out))
        if tags is not None:
            tags_crf = tags.clone()
            tags_crf[tags_crf == -1] = 0
            return -self.crf(emissions, tags_crf, mask=attention_mask, reduction='mean')
        return self.crf.decode(emissions, mask=attention_mask)

In [19]:
# ============================================================
# BLOCK 9C: Improved BERT PubMedBERT + BiLSTM-CRF
# ============================================================

class BERTBiLSTMCRF(nn.Module):
    def __init__(self, num_tags, bert_model_name="bert-base-cased",
                 hidden_dim=512, dropout=0.1):
        super().__init__()
        self.bert    = AutoModel.from_pretrained(bert_model_name)
        bert_dim     = self.bert.config.hidden_size   # 768 for base models
        self.dropout = nn.Dropout(dropout)
        self.bilstm  = nn.LSTM(bert_dim, hidden_dim // 2,
                               num_layers=1, batch_first=True, bidirectional=True)
        self.fc      = nn.Linear(hidden_dim, num_tags)
        self.crf     = CRF(num_tags, batch_first=True)

    def forward(self, input_ids, attention_mask, tags=None):
        bert_out   = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        seq_output = self.dropout(bert_out.last_hidden_state)
        lstm_out, _= self.bilstm(seq_output)
        emissions  = self.fc(self.dropout(lstm_out))
        att_bool   = attention_mask.bool()
        if tags is not None:
            tags_crf = tags.clone()
            tags_crf[tags_crf == -1] = 0
            return -self.crf(emissions, tags_crf, mask=att_bool, reduction='mean')
        return self.crf.decode(emissions, mask=att_bool)

def get_layerwise_optimizer(model, base_lr=3e-5, decay=0.95, wd=1e-2):
    """Build param groups with exponentially decaying LR per BERT layer."""
    no_decay = {"bias", "LayerNorm.weight"}
    num_layers = model.bert.config.num_hidden_layers   # 12 for base

    groups = []
    for name, param in model.bert.named_parameters():
        if not param.requires_grad:
            continue

        if "embeddings" in name:
            layer_lr = base_lr * (decay ** (num_layers + 1))
        else:
            m = re.search(r"layer\.([0-9]+)\.", name)
            depth = int(m.group(1)) if m else num_layers
            layer_lr = base_lr * (decay ** (num_layers - depth))
        wd_val = 0.0 if any(nd in name for nd in no_decay) else wd
        groups.append({"params": [param], "lr": layer_lr, "weight_decay": wd_val})

    other_params = [p for n, p in model.named_parameters()
                    if not n.startswith("bert.") and p.requires_grad]
    groups.append({"params": other_params, "lr": base_lr * 10, "weight_decay": wd})
    return AdamW(groups)

In [20]:
# ============================================================
# BLOCK 10: Generic Train / Evaluate Functions
# ============================================================

def train_epoch(model, loader, optimizer, scheduler=None,
                clip=5.0, is_bert=False, is_elmo=False,
                accum_steps=1, scaler=None):
    model.train()
    total_loss = 0
    optimizer.zero_grad()

    for step, batch in enumerate(loader):
        if is_bert:
            input_ids, att_mask, tags = [b.to(DEVICE) for b in batch]
            with autocast(enabled=USE_AMP):
                loss = model(input_ids, att_mask, tags)
        elif is_elmo:
            ids, tags, att_mask, _ = [b.to(DEVICE) for b in batch]
            with autocast(enabled=USE_AMP):
                loss = model(ids, att_mask, tags)
        else:
            # Static: ids, char_ids, tags, att_mask, mask_pad
            ids, char_ids, tags, att_mask, _ = [b.to(DEVICE) for b in batch]
            with autocast(enabled=USE_AMP):
                loss = model(ids, char_ids, att_mask, tags)

        loss = loss / accum_steps
        if scaler:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        if (step + 1) % accum_steps == 0:
            if scaler:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), clip)
                scaler.step(optimizer)
                scaler.update()
            else:
                nn.utils.clip_grad_norm_(model.parameters(), clip)
                optimizer.step()
            if scheduler:
                scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item() * accum_steps

    return total_loss / len(loader)


def evaluate(model, loader, idx2tag, is_bert=False, is_elmo=False):

    model.eval()
    all_preds, all_true = [], []

    with torch.no_grad():
        for batch in loader:
            if is_bert:
                input_ids, att_mask, tags = [b.to(DEVICE) for b in batch]
                preds    = model(input_ids, att_mask)
                tags_np  = tags.cpu().numpy()
                eval_mask = None
            elif is_elmo:
                ids, tags, att_mask, mask_pad = [b.to(DEVICE) for b in batch]
                preds    = model(ids, att_mask)
                tags_np  = tags.cpu().numpy()
                eval_mask = mask_pad.cpu().numpy()
            else:
                ids, char_ids, tags, att_mask, mask_pad = [b.to(DEVICE) for b in batch]
                preds    = model(ids, char_ids, att_mask)
                tags_np  = tags.cpu().numpy()
                eval_mask = mask_pad.cpu().numpy()

            for i, (pred_seq, true_seq) in enumerate(zip(preds, tags_np)):
                p_tags, t_tags = [], []
                for j, (p, t) in enumerate(zip(pred_seq, true_seq)):
                    if t == -1:
                        continue

                    if eval_mask is not None and not eval_mask[i, j]:
                        continue
                    p_tags.append(idx2tag[p])
                    t_tags.append(idx2tag[t])
                all_preds.append(p_tags)
                all_true.append(t_tags)

    f1 = f1_score(all_true, all_preds)
    return f1, all_preds, all_true


def run_pipeline(name, model, train_loader, val_loader, test_loader,
                 epochs=15, optimizer=None, lr=1e-3,
                 is_bert=False, is_elmo=False,
                 use_scheduler=True, accum_steps=1):
    print(f"\n{'='*60}")
    print(f"Pipeline: {name}")
    print(f"{'='*60}")

    if optimizer is None:
        optimizer = AdamW(model.parameters(), lr=lr, weight_decay=1e-2)

    scheduler, scaler = None, None
    if use_scheduler:
        total_steps = (len(train_loader) // accum_steps) * epochs
        warmup      = max(1, total_steps // 10)
        scheduler   = get_cosine_schedule_with_warmup(
            optimizer, num_warmup_steps=warmup,
            num_training_steps=total_steps)
    if USE_AMP:
        scaler = GradScaler()

    best_val_f1, best_state = 0, None
    for epoch in range(1, epochs + 1):
        t0   = time.time()
        loss = train_epoch(model, train_loader, optimizer, scheduler,
                           is_bert=is_bert, is_elmo=is_elmo,
                           accum_steps=accum_steps, scaler=scaler)
        val_f1, _, _ = evaluate(model, val_loader, idx2tag,
                                is_bert=is_bert, is_elmo=is_elmo)
        print(f"  Epoch {epoch:2d} | loss={loss:.4f} | val_F1={val_f1:.4f} | {time.time()-t0:.1f}s")
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state  = {k: v.clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    test_f1, preds, trues = evaluate(model, test_loader, idx2tag,
                                     is_bert=is_bert, is_elmo=is_elmo)
    print(f"\n  ✅ Test F1 = {test_f1:.4f}")
    print(classification_report(trues, preds))
    safe_name = f"{name[0]}_{name[1]}".replace('/', '_').replace(' ', '_')
    report_path = f"/kaggle/working/NER_Results/report_{safe_name}.txt"
    os.makedirs(os.path.dirname(report_path), exist_ok=True)
    with open(report_path, "w") as rf:
        rf.write(f"Test F1: {test_f1:.4f}\n\n")
        rf.write(classification_report(trues, preds))
    return test_f1

# Step 5: Advanced Optimization for PubMedBERT Fine-Tuning
---
To fine-tune **PubMedBERT** efficiently without running out of GPU memory or breaking the pre-trained weights, we implemented two techniques:

1. **Automatic Mixed Precision (AMP):** Uses `float16` scaling to speed up training on the Kaggle T4 GPU and reduce memory overhead without losing model accuracy.
2. **Layer-wise Learning Rate Decay (LLRD):** Instead of updating the entire network with one learning rate, we apply an exponential decay factor ($\gamma = 0.95$) down through the transformer layers. Deeper layers adapt actively to our specific task, while early layers preserve foundational language features.

In [21]:
# ============================================================
# BLOCK 11: Run All Pipelines
# ============================================================
BATCH_SIZE  = 32
HIDDEN_DIM  = 256 
EPOCHS      = 30 
BERT_EPOCHS = 20
BERT_LR     = 3e-5 
LR          = 1e-3
ACCUM_STEPS = 2 

results = {}
TOKENIZERS = ["whitespace", "bpe", "nltk"]

# -------- STATIC EMBEDDINGS (Word2Vec, GloVe, FastText) ----------
STATIC_EMB = {
    "Word2Vec": w2v_matrix,
    "GloVe"   : glove_matrix,
    "FastText" : ft_matrix,
}

for emb_name, emb_matrix in STATIC_EMB.items():
    for tok_name in TOKENIZERS:
        key = (emb_name, tok_name)
        print(f"\n>>> {emb_name} + {tok_name}")

        tr_ds = NERDataset(train_data, word2idx, tag2idx, char2idx, tok_name)
        va_ds = NERDataset(val_data,   word2idx, tag2idx, char2idx, tok_name)
        te_ds = NERDataset(test_data,  word2idx, tag2idx, char2idx, tok_name)

        tr_dl = DataLoader(tr_ds, BATCH_SIZE, shuffle=True,  collate_fn=collate_fn)
        va_dl = DataLoader(va_ds, BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
        te_dl = DataLoader(te_ds, BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

        model = BiLSTMCRF(
            vocab_size      = VOCAB_SIZE,
            embed_dim       = EMBED_DIM,
            hidden_dim      = HIDDEN_DIM,
            num_tags        = NUM_TAGS,
            pad_idx         = PAD_IDX,
            char_vocab_size = CHAR_VOCAB_SIZE,
            embedding_matrix= emb_matrix,
            freeze_embed    = False,
            dropout         = 0.5,
        ).to(DEVICE)

        f1 = run_pipeline(key, model, tr_dl, va_dl, te_dl,
                          epochs=EPOCHS, lr=LR, use_scheduler=True)
        results[key] = f1
        import json
        os.makedirs("/kaggle/working/NER_Results", exist_ok=True)
        with open("/kaggle/working/NER_Results/results_checkpoint.json", "w") as fp:
            json.dump({str(k): v for k, v in results.items()}, fp)

# -------- ELMo ----------
print("\n>>> ELMo (all tokenizers)")
for tok_name in TOKENIZERS:
    key = ("ELMo", tok_name)
    tr_ds = ELMoNERDataset(train_data, tag2idx, tok_name)
    va_ds = ELMoNERDataset(val_data,   tag2idx, tok_name)
    te_ds = ELMoNERDataset(test_data,  tag2idx, tok_name)
    tr_dl = DataLoader(tr_ds, 16, shuffle=True,  collate_fn=elmo_collate_fn)
    va_dl = DataLoader(va_ds, 16, shuffle=False, collate_fn=elmo_collate_fn)
    te_dl = DataLoader(te_ds, 16, shuffle=False, collate_fn=elmo_collate_fn)
    model = PrecomputedBiLSTMCRF(ELMO_DIM, 512, NUM_TAGS, dropout=0.5).to(DEVICE)
    f1 = run_pipeline(key, model, tr_dl, va_dl, te_dl,
                      epochs=EPOCHS, lr=LR, is_elmo=True, use_scheduler=True)
    results[key] = f1
    import json
    os.makedirs("/kaggle/working/NER_Results", exist_ok=True)
    with open("/kaggle/working/NER_Results/results_checkpoint.json", "w") as fp:
        json.dump({str(k): v for k, v in results.items()}, fp)


>>> Word2Vec + whitespace

Pipeline: ('Word2Vec', 'whitespace')
  Epoch  1 | loss=13.2943 | val_F1=0.0000 | 20.6s
  Epoch  2 | loss=4.9382 | val_F1=0.4488 | 16.9s
  Epoch  3 | loss=3.3445 | val_F1=0.6406 | 16.8s
  Epoch  4 | loss=2.1357 | val_F1=0.6961 | 16.5s
  Epoch  5 | loss=1.4888 | val_F1=0.7135 | 16.7s
  Epoch  6 | loss=1.1511 | val_F1=0.7298 | 16.9s
  Epoch  7 | loss=0.9419 | val_F1=0.7424 | 16.9s
  Epoch  8 | loss=0.8284 | val_F1=0.7429 | 16.9s
  Epoch  9 | loss=0.6767 | val_F1=0.7478 | 17.1s
  Epoch 10 | loss=0.6297 | val_F1=0.7503 | 17.2s
  Epoch 11 | loss=0.5561 | val_F1=0.7497 | 17.2s
  Epoch 12 | loss=0.4599 | val_F1=0.7495 | 17.1s
  Epoch 13 | loss=0.4114 | val_F1=0.7495 | 17.2s
  Epoch 14 | loss=0.3611 | val_F1=0.7455 | 17.2s
  Epoch 15 | loss=0.3384 | val_F1=0.7458 | 17.2s
  Epoch 16 | loss=0.3038 | val_F1=0.7477 | 17.5s
  Epoch 17 | loss=0.2874 | val_F1=0.7525 | 17.4s
  Epoch 18 | loss=0.2438 | val_F1=0.7453 | 17.1s
  Epoch 19 | loss=0.2318 | val_F1=0.7488 | 16.9s
  E

I0000 00:00:1780384340.148325     426 cuda_dnn.cc:529] Loaded cuDNN version 91002


  Done. 4560 sentences embedded.
  Running ELMo on 4581 sentences... (tokenizer=whitespace)
  Done. 4581 sentences embedded.
  Running ELMo on 4797 sentences... (tokenizer=whitespace)
  Done. 4797 sentences embedded.

Pipeline: ('ELMo', 'whitespace')
  Epoch  1 | loss=8.2748 | val_F1=0.6303 | 26.7s
  Epoch  2 | loss=2.3664 | val_F1=0.6938 | 26.2s
  Epoch  3 | loss=1.8217 | val_F1=0.7399 | 26.4s
  Epoch  4 | loss=1.4807 | val_F1=0.7635 | 26.6s
  Epoch  5 | loss=1.1676 | val_F1=0.7784 | 26.3s
  Epoch  6 | loss=0.9025 | val_F1=0.7764 | 26.5s
  Epoch  7 | loss=0.7450 | val_F1=0.7695 | 26.6s
  Epoch  8 | loss=0.5884 | val_F1=0.7494 | 26.3s
  Epoch  9 | loss=0.5033 | val_F1=0.7898 | 26.8s
  Epoch 10 | loss=0.4248 | val_F1=0.7878 | 26.5s
  Epoch 11 | loss=0.3523 | val_F1=0.7872 | 26.5s
  Epoch 12 | loss=0.2888 | val_F1=0.7876 | 26.0s
  Epoch 13 | loss=0.2573 | val_F1=0.7859 | 27.2s
  Epoch 14 | loss=0.2345 | val_F1=0.7882 | 26.8s
  Epoch 15 | loss=0.1871 | val_F1=0.7940 | 26.5s
  Epoch 16 | l

In [24]:
# ============================================================
# BLOCK 11-Setup: BERT-Only Configuration
# ============================================================

# Training Hyperparameters
BATCH_SIZE  = 16
BERT_EPOCHS = 15
BERT_LR     = 3e-5
ACCUM_STEPS = 2
MAX_LEN     = 128

if 'results' not in locals():
    results = {}

TOKENIZERS = ["whitespace", "bpe", "nltk"]

print("PubMedBERT configuration initialized. Preparing for training...")

BERT_CONFIGS = {
    "PubMedBERT" : "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext",
}

for model_label, ckpt in BERT_CONFIGS.items():
    print(f"\n>>> Loading tokenizer for {model_label} ({ckpt})")
    tok = AutoTokenizer.from_pretrained(ckpt)

    for tok_name in TOKENIZERS:
        key = (model_label, tok_name)
        print(f"\n>>> {model_label} + {tok_name}")

        tr_ds = BERTNERDataset(train_data, tag2idx, tok, tok_name)
        va_ds = BERTNERDataset(val_data,   tag2idx, tok, tok_name)
        te_ds = BERTNERDataset(test_data,  tag2idx, tok, tok_name)

        tr_dl = DataLoader(tr_ds, 16, shuffle=True,  collate_fn=bert_collate_fn)
        va_dl = DataLoader(va_ds, 16, shuffle=False, collate_fn=bert_collate_fn)
        te_dl = DataLoader(te_ds, 16, shuffle=False, collate_fn=bert_collate_fn)

        model = BERTBiLSTMCRF(NUM_TAGS, bert_model_name=ckpt,
                               hidden_dim=512, dropout=0.1).to(DEVICE)

        # Layer-wise decayed optimizer for BERT
        opt = get_layerwise_optimizer(model, base_lr=BERT_LR)

        f1 = run_pipeline(key, model, tr_dl, va_dl, te_dl,
                          epochs=BERT_EPOCHS, optimizer=opt,
                          is_bert=True, use_scheduler=True,
                          accum_steps=ACCUM_STEPS)
        results[key] = f1
        
        import json
        import os
        os.makedirs("/kaggle/working/NER_Results", exist_ok=True)
        with open("/kaggle/working/NER_Results/results_checkpoint.json", "w") as fp:
            json.dump({str(k): v for k, v in results.items()}, fp)

PubMedBERT configuration initialized. Preparing for training...

>>> Loading tokenizer for PubMedBERT (microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext)


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]


>>> PubMedBERT + whitespace


...

BertModel LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Pipeline: ('PubMedBERT', 'bpe')
  Epoch  1 | loss=27.9483 | val_F1=0.5634 | 106.0s
  Epoch  2 | loss=6.5757 | val_F1=0.5835 | 106.5s
  Epoch  3 | loss=4.1901 | val_F1=0.7141 | 106.2s
  Epoch  4 | loss=2.8236 | val_F1=0.7609 | 106.4s
  Epoch  5 | loss=1.9346 | val_F1=0.7698 | 106.3s
  Epoch  6 | loss=1.3321 | val_F1=0.7754 | 107.0s
  Epoch  7 | loss=0.8709 | val_F1=0.7877 | 106.3s
  Epoch  8 | loss=0.7047 | val_F1=0.7908 | 106.3s
  Epoch  9 | loss=0.5126 | val_F1=0.7959 | 106.3s
  Epoch 10 | loss=0.3530 | val_F1=0.7928 | 106.0s
  Epoch 11 | loss=0.2856 | val_F1=0.8053 | 106.2s
  Epoch 12 | loss=0.2214 | val_F1=0.8060 | 106.2s
  Epoch 13 | loss=0.1854 | val_F1=0.8079 | 106.1s
  Epoch 14 | loss=0.1510 | val_F1=0.8080 | 105.9s
  Epoch 15 | loss=0.1296 | val_F1=0.8078 | 106.0s

  ✅ Test F1 = 0.7969
              precision    recall  f1-score   support

     Disease       0.78      0.81      0.80     12270

   micro avg       0.78      0.81      0.80     12270
   macro avg       0.78      0

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Pipeline: ('PubMedBERT', 'nltk')
  Epoch  1 | loss=12.9715 | val_F1=0.6646 | 104.8s
  Epoch  2 | loss=1.8013 | val_F1=0.8009 | 104.7s
  Epoch  3 | loss=0.9889 | val_F1=0.8209 | 104.7s
  Epoch  4 | loss=0.5892 | val_F1=0.8353 | 104.8s
  Epoch  5 | loss=0.3399 | val_F1=0.8319 | 104.6s
  Epoch  6 | loss=0.2420 | val_F1=0.8372 | 104.5s
  Epoch  7 | loss=0.1591 | val_F1=0.8417 | 104.5s
  Epoch  8 | loss=0.1072 | val_F1=0.8428 | 104.6s
  Epoch  9 | loss=0.0812 | val_F1=0.8430 | 104.3s
  Epoch 10 | loss=0.0652 | val_F1=0.8425 | 104.4s
  Epoch 11 | loss=0.0274 | val_F1=0.8400 | 104.4s
  Epoch 12 | loss=0.0337 | val_F1=0.8423 | 104.3s
  Epoch 13 | loss=0.0083 | val_F1=0.8432 | 104.2s
  Epoch 14 | loss=0.0183 | val_F1=0.8412 | 105.0s
  Epoch 15 | loss=0.0105 | val_F1=0.8428 | 104.8s

  ✅ Test F1 = 0.8417
              precision    recall  f1-score   support

     Disease       0.82      0.86      0.84      4483

   micro avg       0.82      0.86      0.84      4483
   macro avg       0.82      

In [35]:
# ============================================================
# BLOCK 12: Results Table — STATIC & ELMo ONLY
# ============================================================
import pandas as pd

# Filter only for the non-BERT models
static_elmo_keys = [k for k in results.keys() if k[0] in ["Word2Vec", "GloVe", "FastText", "ELMo"]]

if not static_elmo_keys:
    print(" No Static or ELMo results found in the 'results' dictionary.")
else:
    rows = []
    for key in static_elmo_keys:
        rows.append({"Embedding": key[0], "Tokenizer": key[1], "F1-Score": round(results[key], 4)})

    df_static = pd.DataFrame(rows)
    pivot_static = df_static.pivot(index="Embedding", columns="Tokenizer", values="F1-Score")

    # Order columns consistently
    cols = [c for c in ["whitespace", "bpe", "nltk"] if c in pivot_static.columns]
    pivot_static = pivot_static[cols]

    print("\n" + "="*65)
    print("      SUMMARY: STATIC & ELMo PIPELINES")
    print("="*65)
    print(pivot_static.to_string())
    print("="*65)


      SUMMARY: STATIC & ELMo PIPELINES
Tokenizer  whitespace     bpe    nltk
Embedding                            
ELMo           0.7965  0.7392  0.7864
FastText       0.7578  0.7086  0.7321
GloVe          0.7746  0.7190  0.7414
Word2Vec       0.7470  0.6996  0.7201


In [36]:
# ============================================================
# BLOCK 13: Save Report — STATIC & ELMo ONLY
# ============================================================
if 'pivot_static' in locals():
    out_dir = "/kaggle/working/NER_Results"
    os.makedirs(out_dir, exist_ok=True)
    pivot_static.to_csv(f"{out_dir}/f1_static_elmo.csv")

    with open(f"{out_dir}/report_static_elmo.md", "w") as f:
        f.write("# Static & ELMo NER Results\n\n" + pivot_static.to_markdown())
    print(f"✅ Static/ELMo report saved to {out_dir}")
else:
    print("Skip saving: No static results to process.")

✅ Static/ELMo report saved to /kaggle/working/NER_Results


In [37]:
# ============================================================
# BLOCK 14: Results Table — BERT
# ============================================================
import pandas as pd

# Filter strictly for PubMedBERT results
bert_keys = [k for k in results.keys() if k[0] == "PubMedBERT"]

if not bert_keys:
    print("No PubMedBERT results found in the 'results' dictionary.")
else:
    rows = []
    for key in bert_keys:
        rows.append({"Embedding": key[0], "Tokenizer": key[1], "F1-Score": round(results[key], 4)})

    df_bert = pd.DataFrame(rows)
    pivot_bert = df_bert.pivot(index="Embedding", columns="Tokenizer", values="F1-Score")

    # Order columns consistently
    cols = [c for c in ["whitespace", "bpe", "nltk"] if c in pivot_bert.columns]
    pivot_bert = pivot_bert[cols]

    print("\n" + "="*65)
    print("      SUMMARY: PubMedBERT PIPELINE RESULTS")
    print("="*65)
    print(pivot_bert.to_string())
    print("="*65)


      SUMMARY: PubMedBERT PIPELINE RESULTS
Tokenizer   whitespace     bpe    nltk
Embedding                             
PubMedBERT      0.8533  0.7969  0.8417


In [38]:
# ============================================================
# BLOCK 15: Save Report — BERT ONLY
# ============================================================
if 'pivot_bert' in locals():
    out_dir = "/kaggle/working/NER_Results"
    os.makedirs(out_dir, exist_ok=True)
    
    # Updated file names to reflect the specific model choice
    pivot_bert.to_csv(f"{out_dir}/f1_pubmedbert_only.csv")

    with open(f"{out_dir}/report_pubmedbert_only.md", "w") as f:
        f.write("# PubMedBERT NER Results (By Tokenizer)\n\n" + pivot_bert.to_markdown())
    print(f"✅ PubMedBERT report saved to {out_dir}")
else:
    print("Skip saving: No PubMedBERT results to process.")

✅ PubMedBERT report saved to /kaggle/working/NER_Results


In [39]:
# BLOCK 16: Combined Final Summary
all_rows = []
for key, f1 in results.items():
    all_rows.append({"Model": key[0], "Tokenizer": key[1], "Test F1": round(f1, 4)})

df_all = pd.DataFrame(all_rows).sort_values("Test F1", ascending=False)
print("\n" + "="*65)
print("          FINAL COMBINED RESULTS (All Models)")
print("="*65)
print(df_all.to_string(index=False))
print("="*65)
os.makedirs("/kaggle/working/NER_Results", exist_ok=True)
df_all.to_csv("/kaggle/working/NER_Results/f1_all_models.csv", index=False)


          FINAL COMBINED RESULTS (All Models)
     Model  Tokenizer  Test F1
PubMedBERT whitespace   0.8533
PubMedBERT       nltk   0.8417
PubMedBERT        bpe   0.7969
      ELMo whitespace   0.7965
      ELMo       nltk   0.7864
     GloVe whitespace   0.7746
  FastText whitespace   0.7578
  Word2Vec whitespace   0.7470
     GloVe       nltk   0.7414
      ELMo        bpe   0.7392
  FastText       nltk   0.7321
  Word2Vec       nltk   0.7201
     GloVe        bpe   0.7190
  FastText        bpe   0.7086
  Word2Vec        bpe   0.6996


# Step 6: Summary of Results & Conclusions
---

### Key Findings
* **PubMedBERT was the clear winner (Top F1: 0.8533):** Domain-specific pre-training on biomedical papers gives it a massive advantage over models trained on general text (like GloVe or Word2Vec).
* **Contextual embeddings matter:** ELMo easily outperformed all static embeddings because it calculates representations dynamically based on the sentence context.

### The Tokenizer Takeaway
Interestingly, **BPE subword tokenizers consistently underperformed Whitespace and NLTK**. 

For entity extraction tasks like NER, fragmentation is a major hurdle. When BPE splits technical medical terms into multiple subwords, it creates artificial token boundaries. This complicates the sequence structure for the CRF layer and hurts final boundary detection accuracy. 

**Conclusion:** For clinical NER tasks, pairing a domain-specific model (**PubMedBERT**) with a conservative, word-preserving tokenizer (**Whitespace/NLTK**) yields the cleanest results.